In [ ]:
import requests

colab_ip = requests.get("https://api.ipify.org").text
print("Colab public IP:", colab_ip)

Colab public IP: 34.48.225.115


Cell 1: Install packages


In [ ]:
!pip install -q pandas sqlalchemy pyodbc

In [ ]:
!curl -sSL -O https://packages.microsoft.com/config/ubuntu/$(grep VERSION_ID /etc/os-release | cut -d '"' -f 2)/packages-microsoft-prod.deb
!sudo dpkg -i packages-microsoft-prod.deb
!rm packages-microsoft-prod.deb
!sudo apt-get update -qq
!sudo ACCEPT_EULA=Y apt-get install -y msodbcsql18 unixodbc-dev -qq

(Reading database ... 118462 files and directories currently installed.)
Preparing to unpack packages-microsoft-prod.deb ...
Unpacking packages-microsoft-prod (1.0-ubuntu22.04.1) over (1.0-ubuntu22.04.1) ...
Setting up packages-microsoft-prod (1.0-ubuntu22.04.1) ...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [ ]:
import pyodbc
print(pyodbc.drivers())

['ODBC Driver 18 for SQL Server']


CELL-2 : UPLOAD CSV FILES

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving MedicalHistory.csv to MedicalHistory.csv
Saving Patient.csv to Patient.csv
Saving Doctor.csv to Doctor.csv


In [ ]:
import pandas as pd

doctor_df = pd.read_csv("Doctor.csv")
patient_df = pd.read_csv("Patient.csv")
history_df = pd.read_csv("MedicalHistory.csv")

print("Doctor:", doctor_df.shape)
print("Patient:", patient_df.shape)
print("MedicalHistory:", history_df.shape)

Doctor: (500, 9)
Patient: (10000, 9)
MedicalHistory: (40000, 13)


Cell 3: Check the data BEFORE touching Azure

In [ ]:
print("Doctor columns:")
print(doctor_df.columns.tolist())

print("\nPatient columns:")
print(patient_df.columns.tolist())

print("\nMedicalHistory columns:")
print(history_df.columns.tolist())

print("Duplicate Doctor IDs:",
      doctor_df["DoctorId"].duplicated().sum())

print("Duplicate Patient IDs:",
      patient_df["PatientId"].duplicated().sum())

print("Duplicate History IDs:",
      history_df["HistoryId"].duplicated().sum())

invalid_patient_doctors = ~patient_df["DoctorId"].isin(
    doctor_df["DoctorId"]
)

invalid_history_patients = ~history_df["PatientId"].isin(
    patient_df["PatientId"]
)

print(
    "Patients with invalid DoctorId:",
    invalid_patient_doctors.sum()
)

print(
    "Medical histories with invalid PatientId:",
    invalid_history_patients.sum()
)


Doctor columns:
['DoctorId', 'FirstName', 'LastName', 'Specialization', 'Email', 'Phone', 'CreatedAt', 'UpdatedAt', 'IsDeleted']

Patient columns:
['PatientId', 'FirstName', 'LastName', 'DateOfBirth', 'Gender', 'DoctorId', 'CreatedAt', 'UpdatedAt', 'IsDeleted']

MedicalHistory columns:
['HistoryId', 'PatientId', 'SugarLevel', 'HasDiabetes', 'RecordedAt', 'UpdatedAt', 'BloodPressureSys', 'BloodPressureDia', 'HeartRate', 'Cholesterol', 'BMI', 'Smoking', 'AlcoholConsumption']
Duplicate Doctor IDs: 0
Duplicate Patient IDs: 0
Duplicate History IDs: 0
Patients with invalid DoctorId: 0
Medical histories with invalid PatientId: 0


Cell 4: Azure SQL connection

In [ ]:
import urllib.parse
from sqlalchemy import create_engine

SERVER = "projectehip.database.windows.net"
DATABASE = "EHIP_1ST_DB"

USERNAME = "projectehip"
PASSWORD = "shivam_444"

connection_string = (
    "DRIVER={ODBC Driver 18 for SQL Server};"
    f"SERVER={SERVER},1433;"
    f"DATABASE={DATABASE};"
    f"UID={USERNAME};"
    f"PWD={PASSWORD};"
    "Encrypt=yes;"
    "TrustServerCertificate=no;"
    "Connection Timeout=30;"
)

params = urllib.parse.quote_plus(connection_string)

engine = create_engine(
    f"mssql+pyodbc:///?odbc_connect={params}",
    fast_executemany=True
)

print("Engine created successfully.")

Engine created successfully.


Cell 5: Test connection

In [ ]:
from sqlalchemy import text

with engine.connect() as connection:
    result = connection.execute(
        text("SELECT DB_NAME() AS DatabaseName")
    )

    print(result.fetchone())

('EHIP_1ST_DB',)


Cell 6: Load Doctor

In [ ]:
from sqlalchemy import text

with engine.begin() as connection:  # begin() = single transaction, single connection
    # 1. Turn ON identity insert for this table
    connection.execute(text("SET IDENTITY_INSERT dbo.Doctor ON"))

    # 2. Insert using the SAME connection (not engine directly)
    doctor_df.to_sql(
        "Doctor",
        con=connection,
        schema="dbo",
        if_exists="append",
        index=False,
        chunksize=1000
    )

    # 3. Turn OFF identity insert
    connection.execute(text("SET IDENTITY_INSERT dbo.Doctor OFF"))

print("Doctor data loaded.")

# verify count separately
with engine.connect() as connection:
    result = connection.execute(text("SELECT COUNT(*) FROM dbo.Doctor"))
    print("Doctor count:", result.scalar())

Doctor data loaded.
Doctor count: 500


LOAD Patient and MedicalHistory

In [ ]:
tables_to_load = [
    ("Patient", patient_df),
    ("MedicalHistory", history_df),
]

for table_name, df in tables_to_load:
    with engine.begin() as connection:
        connection.execute(text(f"SET IDENTITY_INSERT dbo.{table_name} ON"))
        df.to_sql(table_name, con=connection, schema="dbo", if_exists="append", index=False, chunksize=1000)
        connection.execute(text(f"SET IDENTITY_INSERT dbo.{table_name} OFF"))
    print(f"{table_name} data loaded.")

Patient data loaded.
MedicalHistory data loaded.
